In [ ]:
import os
from collections import defaultdict
from typing import Any

import numpy as np
from openai import OpenAI


In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook: export OPENAI_API_KEY=sk-...")

client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")

print(f"OpenAI ready. Chat model: {CHAT_MODEL}; embedding model: {EMBED_MODEL}")


In [ ]:
# Same inline K8s corpus used in the hybrid_search notebook, so results are comparable.
CORPUS = [
    {
        "id": "pods",
        "source": "pods.md",
        "text": "A Kubernetes Pod is the smallest deployable unit. Containers in a Pod share network, storage volumes, and lifecycle. Use kubectl describe pod and kubectl logs to debug Pod behavior.",
    },
    {
        "id": "deployment",
        "source": "deployment.md",
        "text": "A Deployment manages ReplicaSets and rolling updates. Use kubectl rollout status deployment/nginx to watch progress and kubectl rollout undo deployment/nginx to roll back a bad release.",
    },
    {
        "id": "service",
        "source": "service.md",
        "text": "A Service gives stable networking for Pods. ClusterIP is internal, NodePort exposes a port on nodes, and LoadBalancer asks the cloud provider for an external endpoint.",
    },
    {
        "id": "probes",
        "source": "probes.md",
        "text": "Readiness probes decide when a Pod can receive traffic. Liveness probes restart stuck containers. Startup probes protect slow-starting applications from premature restarts.",
    },
    {
        "id": "secrets",
        "source": "secrets.md",
        "text": "Kubernetes Secrets store sensitive values such as passwords, tokens, and keys. Enable encryption at rest, restrict RBAC, and avoid printing secret data in logs.",
    },
    {
        "id": "taints",
        "source": "taints.md",
        "text": "Taints repel Pods from nodes. Tolerations allow selected Pods to schedule onto tainted nodes. A NoSchedule taint blocks Pods that lack a matching toleration.",
    },
    {
        "id": "hpa",
        "source": "hpa.md",
        "text": "The HorizontalPodAutoscaler increases or decreases replicas based on CPU, memory, or custom metrics so an application can handle more traffic automatically.",
    },
]

NOISE_DOCS = [
    {"id": "noise-cache", "source": "cache-paper.md", "text": "Cache-aware matrix multiplication improves locality in CPU memory hierarchies and reduces cache misses."},
    {"id": "noise-graph", "source": "graph-paper.md", "text": "Graph partitioning algorithms optimize edge cuts in distributed computation workloads."},
]

ALL_DOCS = CORPUS + NOISE_DOCS
print(f"Inline corpus loaded: {len(CORPUS)} K8s snippets + {len(NOISE_DOCS)} noise snippets")


In [ ]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


def cosine(a: list[float], b: list[float]) -> float:
    av = np.array(a, dtype=float)
    bv = np.array(b, dtype=float)
    denom = np.linalg.norm(av) * np.linalg.norm(bv)
    return float(np.dot(av, bv) / denom) if denom else 0.0


def dense_search_by_vector(query_vector: list[float], docs: list[dict[str, str]], top_k: int = 3) -> list[dict[str, Any]]:
    """Plain cosine search against a single pre-computed query vector."""
    doc_vectors = embed_texts([d["text"] for d in docs])
    ranked = [{**doc, "score": cosine(query_vector, dv)} for doc, dv in zip(docs, doc_vectors)]
    return sorted(ranked, key=lambda d: d["score"], reverse=True)[:top_k]


def dense_search(question: str, docs: list[dict[str, str]], top_k: int = 3) -> list[dict[str, Any]]:
    """Baseline retrieval: embed the raw question and search with it directly."""
    qv = embed_texts([question])[0]
    return dense_search_by_vector(qv, docs, top_k=top_k)


In [ ]:
# HyDE: instead of embedding the raw (sometimes vague) question, ask the LLM to write
# a short, plausible answer first. A fluent answer tends to share more vocabulary and
# phrasing with the real documents than a short, vaguely-worded question does, so
# embedding the *hypothesis* and searching with that often retrieves better matches.

HYDE_SYSTEM_PROMPT = (
    "You are a helpful assistant. Given a user question, write a brief, plausible answer "
    "(2-3 sentences) that would help retrieve relevant documents. Write only the answer, "
    "no preamble."
)


def generate_hypothesis(question: str, temperature: float = 0.7) -> str:
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": HYDE_SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        temperature=temperature,
    )
    return (response.choices[0].message.content or "").strip()


In [ ]:
def hyde_search(
    question: str,
    docs: list[dict[str, str]],
    top_k: int = 3,
    num_hypotheses: int = 3,
) -> list[dict[str, Any]]:
    """Generate a few hypothetical answers, search with each (plus the raw question),
    then merge results and keep the best score per doc."""
    hypotheses = [generate_hypothesis(question) for _ in range(num_hypotheses)]
    search_texts = hypotheses + [question]

    query_vectors = embed_texts(search_texts)
    doc_vectors = embed_texts([d["text"] for d in docs])

    best: dict[str, dict[str, Any]] = {}
    for qv in query_vectors:
        for doc, dv in zip(docs, doc_vectors):
            score = cosine(qv, dv)
            if doc["id"] not in best or score > best[doc["id"]]["score"]:
                best[doc["id"]] = {**doc, "score": score}

    return sorted(best.values(), key=lambda d: d["score"], reverse=True)[:top_k]


In [ ]:
# A deliberately vague question: it shares almost no vocabulary with service.md
# ("Service", "ClusterIP", "networking"), which is the doc that actually answers it.
# Plain embedding similarity on the raw question struggles here; a generated hypothetical
# answer is more likely to use the same domain terms as the real doc.
vague_query = "other pods can't talk to mine even though mine looks totally fine, why"

print("Baseline dense search (raw question embedded directly)")
for c in dense_search(vague_query, ALL_DOCS):
    print(f"  {c['source']:<20} {c['score']:.3f}")


In [ ]:
hypotheses_preview = [generate_hypothesis(vague_query) for _ in range(3)]
for i, h in enumerate(hypotheses_preview, start=1):
    print(f"Hypothesis {i}: {h}\n")


In [ ]:
print("HyDE search (hypothetical answers + raw question, best score per doc)")
for c in hyde_search(vague_query, ALL_DOCS):
    print(f"  {c['source']:<20} {c['score']:.3f}")


In [ ]:
# Sanity check on a query that's already lexically close to its answer doc -- HyDE
# shouldn't make things worse here, it should land on the same top result as baseline.
exact_query = "kubectl rollout undo deployment/nginx"

print("Baseline dense search")
for c in dense_search(exact_query, ALL_DOCS):
    print(f"  {c['source']:<20} {c['score']:.3f}")

print("\nHyDE search")
for c in hyde_search(exact_query, ALL_DOCS):
    print(f"  {c['source']:<20} {c['score']:.3f}")
